In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [2]:
df = pd.read_csv(r"C:\Users\Harshada\Downloads\sensor_Crop_Dataset_Final.csv")
df

,Crop,Variety,Region,Season,Soil_Type,Nitrogen,Phosphorus,Potassium,Temperature,Humidity,pH_Value,Rainfall,Yield
0,Wheat,Soft Red,Madhya Pradesh,Rabi,Clay,69.074766,53.954402,88.067625,17.261834,72.941652,4.631301,302.842639,5633.46
1,Tomato,Beefsteak,Madhya Pradesh,Kharif,Clay,107.329352,70.102134,32.081067,21.846116,99.361954,4.761658,94.693847,32329.90
2,Sugarcane,Co 86032,Tamil Nadu,Kharif,Clay,130.634624,67.204533,28.294252,33.246895,81.506836,6.566007,83.563685,88929.17
3,Sugarcane,Co 0238,Andhra Pradesh,Kharif,Silt,15.169301,87.493181,14.336679,14.396289,59.274465,6.296297,31.508836,76689.66
4,Maize,Sweet,Maharashtra,Kharif,Sandy,21.881965,89.269712,38.833885,16.773218,51.191584,8.268274,295.193482,5385.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,Rice,Arborio,Andhra Pradesh,Kharif,Silt,15.286598,32.026745,52.276522,30.496937,98.813042,7.549344,238.537544,5661.52
19996,Wheat,Durum,Madhya Pradesh,Rabi,Clay,29.790472,17.182611,74.772890,40.974020,83.002347,5.895767,333.470901,4624.75
19997,Wheat,Hard Red,Haryana,Rabi,Loamy,25.001919,19.140862,32.719994,29.001299,55.231845,8.230164,119.351274,4630.44
19998,Wheat,Soft Red,Haryana,Rabi,Loamy,74.396171,42.100129,20.669153,22.349399,84.369830,7.878051,385.969414,6494.85


In [3]:
X = df.drop("Yield", axis=1)
y = df["Yield"]


In [4]:
categorical_cols = [
    "Crop",
    "Variety",
    "Region",
    "Season",
    "Soil_Type"
]
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


In [8]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
}
results = []

best_model = None
best_model_name = ""
best_r2 = -999

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    mae = mean_absolute_error(y_test, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )
results.append([ name, round(r2, 4),round(mae, 2), round(rmse, 2) ])
if r2 > best_r2:
        best_r2 = r2
        best_model = pipe
        best_model_name = name
        results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "R2 Score",
        "MAE",
        "RMSE"
    ]
)

results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
)

print("\nMODEL COMPARISON")
print(results_df)

print("\nBEST MODEL:", best_model_name)
print("BEST R² SCORE:", round(best_r2, 4))


MODEL COMPARISON
     Model  R2 Score     MAE     RMSE
0  XGBoost    0.9969  851.09  1555.72

BEST MODEL: XGBoost
BEST R² SCORE: 0.9969


In [10]:
joblib.dump(
    best_model,
    "crop_yield_model.pkl"
)

print("\nModel Saved Successfully")
print("File: crop_yield_model.pkl")



Model Saved Successfully
File: crop_yield_model.pkl


In [13]:
print("\n========== ENTER DETAILS ==========")

crop = input("Crop: ")
variety = input("Variety: ")
region = input("Region: ")
season = input("Season: ")
soil_type = input("Soil Type: ")

nitrogen = float(input("Nitrogen: "))
phosphorus = float(input("Phosphorus: "))
potassium = float(input("Potassium: "))
temperature = float(input("Temperature: "))
humidity = float(input("Humidity: "))
ph_value = float(input("pH Value: "))
rainfall = float(input("Rainfall: "))


========== ENTER DETAILS ==========


Crop:  Maize
Variety:  Sweet
Region:  Maharashtra
Season:  Rabi
Soil Type:  Loamy
Nitrogen:  84.69
Phosphorus:  45.69
Potassium:  32.48
Temperature:  48.25
Humidity:  88.41
pH Value:  7.42
Rainfall:  140.25


In [14]:
user_data = pd.DataFrame({
    "Crop": [crop],
    "Variety": [variety],
    "Region": [region],
    "Season": [season],
    "Soil_Type": [soil_type],
    "Nitrogen": [nitrogen],
    "Phosphorus": [phosphorus],
    "Potassium": [potassium],
    "Temperature": [temperature],
    "Humidity": [humidity],
    "pH_Value": [ph_value],
    "Rainfall": [rainfall]
})

In [15]:
predicted_yield = best_model.predict(user_data)

print("\n==============================")
print("PREDICTED CROP YIELD")
print("==============================")
print(round(predicted_yield[0], 2), "kg/hectare")


PREDICTED CROP YIELD
6674.38 kg/hectare
